# ATH D1 model comparison on Colab: complete the `qwen3.5:4b` baseline, then run one stronger model under the same investigator

One question: **does changing the local model improve ATH's investigation when the architecture, evidence, prompts, schema, budgets and cases stay fixed?** Both models run `d1-investigator-v3` unchanged through `scripts/local_ablation.py`; the paired reading comes from `scripts/local_compare.py`. Nothing in this notebook tunes a prompt, a budget or a case.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough: the challenger is 6.6 GB and the context is 10K tokens). Run the cells in order. Nothing starts a model until you run the cell that does.

**Checkpoints.** Three cells raise instead of continuing when their check fails: after the baseline (20 validated rows), after the challenger smoke (rows written, structured output parsed), and before the comparison (both sets complete). A raised checkpoint means stop and read, not re-run with a flag.

**Resuming after a disconnect.** Rows are written one file each as they finish and a rerun skips them within a runtime. Between runtimes nothing persists on Colab, so download the results zip (section 8, or the interim download cell in section 4) and upload it in section 1 next time; rows it holds are validated and skipped, not rerun.

**What must never happen here.** No `--ignore-ram-floor` (the guard now credits a model already loaded in VRAM, which is why the earlier stop happened; the warm-up cells load the model before the first row). No token in a URL or an output. No edit to prompts, schema, `ClaimVerifier`, scoring or frozen artifacts to make a case pass. Frozen holdout cases (`reports/m19/`, `reports/m19b/`) are not read.

## 0. Configuration

Everything the later cells read. Change values here, not in the cells.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<private-owner>/agentic-threat-hunter.git"   # no token in the URL; see section 1
BRANCH = "m14-real-data-validation"
EXPECTED_COMMIT = "c3bd08a4e24d5c7bf95c22801f77e7fe4dc9f563"   # the commit this notebook was written for; the checkout must contain it (ancestor check)
REPO = Path("/content/agentic-threat-hunter")

ARM = "D1"
BASELINE_MODEL = "qwen3.5:4b"          # the D1 v3 baseline: 9 rows exist, 11 remain
CHALLENGER_MODEL = "qwen3.5:9b"        # one challenger: same family, dense, Ollama default quantisation (Q4_K_M, 6.6 GB)
EXPORT_TELEMETRY_CACHE = True          # section 8 also zips data/external (~1.6 GB) with checksums, to upload next time and skip the fetch
REPEAT = 1
SEED = 0
EXPECTED_CASES = 20

# The smoke set is the same five cases the 4B smoke used: two malicious, two benign, one unlabelled cloud case.
SMOKE = [
    "dedale_injected_dev:V1/CASE-001", "dedale_injected_dev:V2/CASE-001",
    "dedale_injected_dev:V7/CASE-001", "dedale_injected_dev:V8/CASE-001",
    "flaws_cloud/CASE-071",
]

# No Google Drive. Earlier results and cached telemetry come in as uploaded zips and go out as
# downloaded zips; the local paths below live only for this runtime.
UPLOADS = Path("/content/uploads")                     # drop zips here (or use the upload cell): rows zips, telemetry cache zip
RESULTS = Path("/content/results")                     # what this session produced, zipped for download at the end
TELEMETRY_CACHE = UPLOADS / "telemetry_cache"          # an unpacked telemetry_cache_*.zip from an earlier session, verified before reuse
RESULTS_BASELINE = RESULTS / "results_4b"              # completed 4B rows + freeze + summary, by manifest hash
RESULTS_CHALLENGER = RESULTS / "results_challenger"    # challenger rows + freeze + summary, by manifest hash
SUMMARIES = RESULTS / "summaries"                      # comparisons, environment records

OLLAMA_URL = "http://127.0.0.1:11434"
DEV_DIR = REPO / "reports" / "local" / "dev"

BASELINE_OK = False
SMOKE_OK = False
CHALLENGER_OK = False
for d in (UPLOADS, RESULTS_BASELINE, RESULTS_CHALLENGER, SUMMARIES):
    d.mkdir(parents=True, exist_ok=True)
print("configured:", BASELINE_MODEL, "vs", CHALLENGER_MODEL, "| uploads:", UPLOADS, "| results:", RESULTS)

## 1. Environment: Ollama daemon, uploads from earlier sessions, repository, Python

The daemon is started detached with its log in a file. Anything from an earlier session (the `dev_results.zip` with the nine 4B rows, a `telemetry_cache_*.zip` from a previous export) is uploaded in the second cell; skip the upload when you have nothing. The repository is cloned with a token read from a Colab secret named `GH_TOKEN` through `GIT_ASKPASS`, so the token never appears in a command, an output or `.git/config`.

In [ ]:
import subprocess, time, urllib.request, os, sys, json, shutil

subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True, capture_output=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

def daemon_up() -> bool:
    try:
        urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=2)
        return True
    except Exception:
        return False

if not daemon_up():
    log = open("/content/ollama.log", "ab")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for attempt in range(120):
        if daemon_up():
            break
        time.sleep(1)
    else:
        raise SystemExit("ollama daemon did not start; see /content/ollama.log")
print("ollama", json.load(urllib.request.urlopen(OLLAMA_URL + "/api/version"))["version"], "is up")

In [ ]:
# OPTIONAL. Upload zips from earlier sessions: the dev_results.zip (rows, freeze, summary) and/or a
# telemetry_cache_*.zip exported by section 8. Close the dialog with nothing selected to skip.
# Alternatively drag files into /content/uploads with the Files sidebar and just run this cell.
from google.colab import files
import zipfile
try:
    uploaded = files.upload()
except Exception as exc:  # noqa: BLE001
    uploaded = {}
    print("upload dialog closed or unavailable:", exc)
for name, data in uploaded.items():
    (UPLOADS / name).write_bytes(data)
zips = sorted(UPLOADS.glob("*.zip"))
print(f"{len(zips)} zip(s) in {UPLOADS}:", [z.name for z in zips] or "none (fresh session: everything is fetched and run)")
for z in zips:
    if z.name.startswith("telemetry_cache"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(UPLOADS)
        print("unpacked", z.name, "->", TELEMETRY_CACHE)

In [ ]:
from google.colab import userdata

# The token is handed to git through an askpass helper: never on a command line, never in
# .git/config, never in this output. Colab secret name: GH_TOKEN (read access to this repo).
token = userdata.get("GH_TOKEN")
askpass = Path("/content/askpass.sh")
askpass.write_text("#!/bin/sh\necho \"$GH_TOKEN\"\n")
askpass.chmod(0o700)
env = {**os.environ, "GIT_ASKPASS": str(askpass), "GH_TOKEN": token, "GIT_TERMINAL_PROMPT": "0"}

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True, env=env)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True, env=env)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "lfs", "install", "--skip-repo"], capture_output=True)
subprocess.run(["git", "-C", str(REPO), "lfs", "pull"], env=env, capture_output=True)
del token, env

head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
remote = subprocess.run(["git", "-C", str(REPO), "remote", "get-url", "origin"], capture_output=True, text=True).stdout.strip()
assert "@" not in remote, "the remote URL must not carry a credential"
print("checked out", BRANCH, "at", head[:12])
if EXPECTED_COMMIT:
    ok = subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", EXPECTED_COMMIT, "HEAD"]).returncode == 0
    if not ok:
        raise SystemExit(f"HEAD {head[:12]} does not contain the expected commit {EXPECTED_COMMIT[:12]}; the branch moved backwards or the wrong branch is checked out")
    print("contains expected commit", EXPECTED_COMMIT[:12])
else:
    print("EXPECTED_COMMIT not set: no commit check (set it in section 0 for a reproducible run)")
os.chdir(REPO)

In [ ]:
assert sys.version_info >= (3, 10), f"Python {sys.version.split()[0]} is below the project's 3.10 floor"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
# An editable install registers with the interpreter at start-up; this kernel was already running,
# so put the sources on its path directly (the scripts do the same for themselves).
for p in (REPO / "src", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import ath  # noqa: E402
print("python", sys.version.split()[0], "| ath importable from", Path(ath.__file__).parent)

## 2. Telemetry: restore the verified cache or fetch, then inject and build the dev manifest

`data/external/` (flaws.cloud CloudTrail and the DEDALE Winlogbeat day) is restored from an uploaded `telemetry_cache_*.zip` only when every cached file's size and sha256 match the checksum file written when it was exported; otherwise it is fetched. The injected cases and the dev manifest are **rebuilt every session** (the telemetry digest depends on the runtime); the manifest hash printed here is the identity every row must carry.

In [ ]:
import hashlib

EXTERNAL = REPO / "data" / "external"
CACHE_DATA = TELEMETRY_CACHE / "data_external"
CHECKSUMS = TELEMETRY_CACHE / "CHECKSUMS.json"
CACHED_SUBDIRS = ("flaws_cloud", "dedale")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def checksum_tree(root: Path) -> dict:
    out = {}
    for sub in CACHED_SUBDIRS:
        for p in sorted((root / sub).rglob("*")):
            if p.is_file():
                out[str(p.relative_to(root)).replace("\\", "/")] = {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
    return out

def cache_is_valid() -> bool:
    if not CHECKSUMS.exists() or not CACHE_DATA.exists():
        return False
    recorded = json.loads(CHECKSUMS.read_text())["files"]
    if not recorded:
        return False
    for rel, meta in recorded.items():
        p = CACHE_DATA / rel
        if not p.exists() or p.stat().st_size != meta["bytes"]:
            print("cache invalid: size mismatch or missing", rel); return False
    # sizes match; verify content of a stable sample plus every small file (full hash of 1.6 GB on Drive is slow)
    sample = sorted(recorded)[::max(1, len(recorded) // 25)]
    for rel in sample:
        if sha256_file(CACHE_DATA / rel) != recorded[rel]["sha256"]:
            print("cache invalid: sha256 mismatch", rel); return False
    return True

def have_local() -> bool:
    return all((EXTERNAL / s).exists() and any((EXTERNAL / s).iterdir()) for s in CACHED_SUBDIRS)

if cache_is_valid():
    print("telemetry cache verified; restoring to local disk ...")
    for sub in CACHED_SUBDIRS:
        subprocess.run(["rsync", "-a", str(CACHE_DATA / sub) + "/", str(EXTERNAL / sub) + "/"], check=True)
    restored = checksum_tree(EXTERNAL)
    recorded = json.loads(CHECKSUMS.read_text())["files"]
    bad = [r for r in recorded if restored.get(r, {}).get("sha256") != recorded[r]["sha256"]]
    if bad:
        raise SystemExit(f"restored copy differs from the cache checksums for {len(bad)} file(s), e.g. {bad[:3]}; delete the cache and refetch")
    print(f"restored and verified {len(restored)} file(s)")
elif have_local():
    print("data/external already present on this runtime (fetched earlier in this session)")
else:
    print("no valid cache; fetching (several minutes) ...")
    subprocess.run([sys.executable, "scripts/fetch_external.py", "flaws_cloud"], check=True)
    subprocess.run([sys.executable, "scripts/dedale_fetch_hours.py", "--days", "2"], check=True)

print("telemetry present under", EXTERNAL, "(section 8 can export it as a checksummed cache zip)")

In [ ]:
subprocess.run([sys.executable, "scripts/local_inject_dedale.py"], check=True)
subprocess.run([sys.executable, "scripts/local_manifest.py", "build"], check=True)
MANIFEST = json.loads((DEV_DIR / "MANIFEST.json").read_text())
MANIFEST_HASH = MANIFEST["manifest_hash"]
assert len(MANIFEST["cases"]) == EXPECTED_CASES, f"manifest has {len(MANIFEST['cases'])} cases, expected {EXPECTED_CASES}"
print("dev manifest", MANIFEST_HASH[:12], "at", MANIFEST["head"], "|", len(MANIFEST["cases"]), "cases")
print("rows for this session live under reports/local/dev/rows/<arm>_<model>/m" + MANIFEST_HASH[:12] + "_<prompt version>/")

## 3. GPU readiness

What the machine is, what each model is according to the daemon, and whether it fits. `check_model()` is re-run after each warm-up: it errors when the model is not resident, warns when part of it is not in VRAM, and warns when the free system memory plus the model already loaded is under the guard's floor for that size class.

In [ ]:
GIB = 1024 ** 3

def gpu_report():
    if shutil.which("nvidia-smi") is None:
        print("WARNING: no nvidia-smi; this runtime has no NVIDIA GPU. Both models would run on the CPU (10x slower).")
        return []
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    gpus = []
    for line in out.splitlines():
        name, total, used, free = [p.strip() for p in line.split(",")]
        gpus.append({"name": name, "total_mib": int(total), "used_mib": int(used), "free_mib": int(free)})
        print(f"GPU: {name}  VRAM total {int(total)/1024:.1f} GiB, used {int(used)/1024:.1f} GiB, free {int(free)/1024:.1f} GiB")
    return gpus

def mem_available_bytes():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024

def show(model):
    req = urllib.request.Request(OLLAMA_URL + "/api/show", data=json.dumps({"model": model}).encode(), headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=60))

def loaded():
    return {m["name"]: m for m in json.load(urllib.request.urlopen(OLLAMA_URL + "/api/ps", timeout=10)).get("models", [])}

def warm_up(model):
    """Load the model and keep it resident (the client uses keep_alive=-1 too), so the RAM guard
    sees it before the first row. Loads nothing else and generates nothing."""
    req = urllib.request.Request(OLLAMA_URL + "/api/generate", data=json.dumps({"model": model, "keep_alive": -1}).encode(),
                                 headers={"Content-Type": "application/json"})
    json.load(urllib.request.urlopen(req, timeout=600))

def check_model(model, *, strict=True):
    from ath.evaluation.ablation.local import ram_floor_for
    details = show(model)
    d = details.get("details", {})
    info = details.get("model_info", {})
    ctx_key = next((k for k in info if k.endswith(".context_length")), None)
    print(f"{model}: family {d.get('family')}, parameters {d.get('parameter_size')}, quantisation {d.get('quantization_level')}, "
          f"model context {info.get(ctx_key) if ctx_key else '?'}; client sends num_ctx=10240, num_predict cap 2048 (768 per D1 call), think=False, format=schema, temperature 0, seed 0")
    entry = loaded().get(model)
    if entry is None:
        msg = f"{model} is not loaded; run warm_up('{model}') first"
        if strict: raise SystemExit(msg)
        print("WARNING:", msg); return
    size, vram = int(entry.get("size", 0)), int(entry.get("size_vram", 0))
    where = "entirely in VRAM" if vram >= size else (f"{vram/size:.0%} in VRAM, rest in system memory" if vram else "entirely in system memory (CPU)")
    print(f"  resident: {size/GIB:.2f} GiB, {where}")
    if vram < size:
        print("  WARNING: the model is not fully on the GPU; wall times will not be comparable with a GPU run. Check nvidia-smi and the Ollama log.")
    floor = ram_floor_for(d.get("parameter_size"))
    avail = mem_available_bytes()
    if floor is not None:
        effective = avail + size
        print(f"  RAM guard: {avail/GIB:.2f} GiB available + {size/GIB:.2f} GiB already resident = {effective/GIB:.2f} GiB vs floor {floor/GIB:.1f} GiB -> {'ok' if effective >= floor else 'WOULD REFUSE'}")
        if effective < floor and strict:
            raise SystemExit("the RAM guard would refuse the first row; free memory (restart the runtime) rather than overriding the guard")
    return details

gpus = gpu_report()
print(f"system RAM available: {mem_available_bytes()/GIB:.2f} GiB")
print("models the daemon has:", [m["name"] for m in json.load(urllib.request.urlopen(OLLAMA_URL + "/api/tags"))["models"]] or "none yet")

## 4. Baseline: complete the `qwen3.5:4b` D1 v3 run (20 cases)

Existing rows are restored from the uploaded zips (section 1) into this session's rows directory **without overwriting** anything, then `validate-rows` checks each against the live freeze and manifest: manifest hash, telemetry hash, model digest, daemon version, client configuration, prompt version and investigator hashes. A row that fails is moved to a `.quarantine/` folder, never deleted, and its case is rerun. The run then completes the remaining cases and the checkpoint requires 20 validated rows before anything else may start.

In [ ]:
def run(*args, check=True):
    """scripts/local_ablation.py with the shared settings; output streams to the cell."""
    cmd = [sys.executable, "scripts/local_ablation.py", *args]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def rows_dir_for(model):
    from ath.agent.investigator import D1_PROMPT_VERSION
    import local_ablation as la  # noqa: F401  (scripts on sys.path below)
    return DEV_DIR / "rows" / f"{ARM}_{la.model_slug(model)}" / f"m{MANIFEST_HASH[:12]}_{la.model_slug(D1_PROMPT_VERSION)}"

sys.path.insert(0, str(REPO / "scripts"))
import local_ablation as la
from ath.agent.investigator import D1_PROMPT_VERSION

def restore_rows(model, results_root):
    """Copy any row files for this model + manifest + prompt version from the uploaded zips (any
    dev_results.zip or results export from an earlier session) into the session's rows directory.
    Never overwrites; validate-rows decides afterwards whether a restored row belongs."""
    dest = rows_dir_for(model)
    dest.mkdir(parents=True, exist_ok=True)
    rel = dest.relative_to(DEV_DIR)
    copied = skipped = 0
    # 1. zips uploaded in section 1 (or dropped into /content/uploads)
    for z in sorted(UPLOADS.glob("*.zip")):
        import zipfile
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                if name.endswith(".json") and str(rel).replace("\\", "/") in name and "/rows/" in name:
                    target = dest / Path(name).name
                    if target.exists():
                        skipped += 1; continue
                    target.write_bytes(zf.read(name)); copied += 1
    # 2. rows saved by an earlier session of this notebook
    for src in sorted((results_root / MANIFEST_HASH[:12] / "rows").rglob("*.json")) if (results_root / MANIFEST_HASH[:12] / "rows").exists() else []:
        if str(rel).replace("\\", "/") not in str(src).replace("\\", "/"):
            continue
        target = dest / src.name
        if target.exists():
            skipped += 1; continue
        shutil.copy2(src, target); copied += 1
    print(f"{model}: restored {copied} row file(s) into {rel}, {skipped} already present")
    return dest

baseline_rows = restore_rows(BASELINE_MODEL, RESULTS_BASELINE)
print(sorted(p.name[:60] for p in baseline_rows.glob("*.json")))

In [ ]:
subprocess.run(["ollama", "pull", BASELINE_MODEL], check=True)
run("freeze", "--model", BASELINE_MODEL, "--arm", ARM)
warm_up(BASELINE_MODEL)
check_model(BASELINE_MODEL)
gpu_report()

In [ ]:
# Restored rows must belong to THIS experiment identity or they are set aside (quarantined, not deleted).
result = run("validate-rows", "--model", BASELINE_MODEL, "--arm", ARM, "--quarantine", check=False)
valid_now = len(list(baseline_rows.glob("*.json")))
print(f"{valid_now} row(s) stand for {BASELINE_MODEL}; the run below completes the rest")

In [ ]:
# Resumable: every finished row is a file; a disconnect loses at most the case in flight.
result = run("run", "--model", BASELINE_MODEL, "--arm", ARM, "--repeat", str(REPEAT), "--seed", str(SEED), check=False)
print("run exit code", result.returncode, "(0 done; 3 controlled interruption; other = refused, read the last lines)")
run("summarise", "--model", BASELINE_MODEL, "--arm", ARM, "--repeat", str(REPEAT))

In [ ]:
# CHECKPOINT 1: the baseline is complete and every row is this experiment's.
summary = json.loads((DEV_DIR / f"SUMMARY_{ARM}_{la.model_slug(BASELINE_MODEL)}_rep{REPEAT}.json").read_text())
n_rows = summary["rows_written"]
excluded = summary.get("rows_excluded") or {}
print(f"rows written {n_rows}/{EXPECTED_CASES}; completed strict {summary['completed_strict']}; degraded {summary.get('rows_degraded')}; excluded {excluded}")
print("investigation:", json.dumps({k: summary["investigation"][k] for k in ("link_2_recovered", "link_2_defined", "abstention_rate", "cases_using_new_evidence", "truncated_rows")}, indent=None))
if n_rows != EXPECTED_CASES or excluded:
    raise SystemExit("CHECKPOINT 1 FAILED: the baseline is not complete under this manifest/freeze. Read the run output above. "
                     "Before restarting the runtime, run the interim download cell below so the finished rows come back with you.")
BASELINE_OK = True
print("CHECKPOINT 1 passed: baseline complete")

In [ ]:
def save_results(model, results_root):
    """Collect rows + freeze + summary + manifest + guard events under results_root/<manifest12>/
    (local; zipped for download by the cells that follow). Existing files are never overwritten."""
    target = results_root / MANIFEST_HASH[:12]
    target.mkdir(parents=True, exist_ok=True)
    rows_src = rows_dir_for(model)
    rows_dst = target / "rows" / rows_src.relative_to(DEV_DIR / "rows")
    subprocess.run(["rsync", "-a", "--ignore-existing", str(rows_src) + "/", str(rows_dst) + "/"], check=True)
    q = rows_src.with_name(rows_src.name + ".quarantine")
    if q.exists():
        subprocess.run(["rsync", "-a", "--ignore-existing", str(q) + "/", str(rows_dst.with_name(rows_dst.name + ".quarantine")) + "/"], check=True)
    for name in ("MANIFEST.json", "MANIFEST.md", f"ENVIRONMENT_{la.model_slug(model)}.json",
                 f"SUMMARY_{ARM}_{la.model_slug(model)}_rep{REPEAT}.json", f"SUMMARY_{ARM}_{la.model_slug(model)}_rep{REPEAT}.md"):
        src = DEV_DIR / name
        if src.exists() and not (target / name).exists():
            shutil.copy2(src, target / name)
    n = sum(1 for _ in rows_dst.glob("*.json"))
    print(f"saved {model}: {n} row(s) + records under {target}")

save_results(BASELINE_MODEL, RESULTS_BASELINE)

In [ ]:
# Interim download: the baseline as it stands (rows, freeze, summary, manifest). Run this any time,
# also after a failed checkpoint, so nothing finished is lost when the runtime goes away. Upload the
# zip in section 1 next session and its rows are validated and skipped.
def download_zip(name: str, *sources: Path):
    zip_path = Path(f"/content/{name}.zip")
    if zip_path.exists():
        zip_path.unlink()
    subprocess.run(["zip", "-r", "-q", str(zip_path), *[str(s) for s in sources if Path(s).exists()]], check=True, cwd="/content")
    print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    files.download(str(zip_path))

os.chdir("/content")
download_zip(f"ath_baseline_{MANIFEST_HASH[:12]}", Path("agentic-threat-hunter/reports/local/dev"))
os.chdir(REPO)

## 5. Challenger smoke: compatibility only

The challenger runs the identical investigator on the five smoke cases. This cell answers only: does the model produce the structured output the schema demands, does it reach the tools, does it fit, and are the rows stamped with its own identity. Quality on these five cases is **not** read here and changes nothing.

In [ ]:
if not BASELINE_OK:
    raise SystemExit("run and pass CHECKPOINT 1 first")
subprocess.run(["ollama", "pull", CHALLENGER_MODEL], check=True)
run("freeze", "--model", CHALLENGER_MODEL, "--arm", ARM)
warm_up(CHALLENGER_MODEL)
check_model(CHALLENGER_MODEL)
gpu_report()
env_ch = json.loads((DEV_DIR / f"ENVIRONMENT_{la.model_slug(CHALLENGER_MODEL)}.json").read_text())
env_bl = json.loads((DEV_DIR / f"ENVIRONMENT_{la.model_slug(BASELINE_MODEL)}.json").read_text())
same = {k: env_ch["local"]["client_configuration"].get(k) == env_bl["local"]["client_configuration"].get(k)
        for k in env_ch["local"]["client_configuration"] if k != "model"}
print("client configuration identical to the baseline except the model:", all(same.values()), {k: v for k, v in same.items() if not v} or "")
print("investigator hashes identical:", env_ch["local"]["investigator"] == env_bl["local"]["investigator"])
print("challenger:", env_ch["local"]["model"])

In [ ]:
challenger_rows = restore_rows(CHALLENGER_MODEL, RESULTS_CHALLENGER)
run("validate-rows", "--model", CHALLENGER_MODEL, "--arm", ARM, "--quarantine", check=False)
result = run("run", "--model", CHALLENGER_MODEL, "--arm", ARM, "--repeat", str(REPEAT), "--seed", str(SEED), "--only", *SMOKE, check=False)
print("smoke exit code", result.returncode)
run("summarise", "--model", CHALLENGER_MODEL, "--arm", ARM, "--repeat", str(REPEAT))

In [ ]:
# CHECKPOINT 2: compatibility. Rows exist for every smoke case; structured output parsed; no context refusals.
import glob
smoke_rows = []
for path in sorted(challenger_rows.glob("*.json")):
    payload = json.loads(path.read_text())
    key = f"{payload['key']['corpus']}/{payload['key']['case_id']}"
    if key in SMOKE:
        smoke_rows.append((key, payload))
problems = []
for key, payload in smoke_rows:
    row = payload["row"]; inv = row["state"].get("investigation", {}); llm = row["state"]["llm"]
    ctx = [e for e in llm.get("errors", []) if "exceeds num_ctx" in e]
    print(f"{key}: calls {inv.get('model_calls')} | unparseable {llm.get('unparseable_responses')} | truncated {inv.get('output_truncated')} | "
          f"probes {inv.get('probes_run')} | degraded {row.get('llm_degraded')} | model {payload['header']['model']['model']} {str(payload['header']['model']['digest'])[:12]}")
    for r in inv.get("rounds", []):
        print("    round", r.get("round"), "| chose", (r.get("chosen_probe") or {}).get("tool"), "| raw:", (r.get("raw") or "")[:160].replace("\n", " "))
    if llm.get("unparseable_responses"):
        problems.append(f"{key}: {llm['unparseable_responses']} reply(ies) were not the JSON the schema demands")
    if ctx:
        problems.append(f"{key}: context refusal {ctx[0][:80]}")
    if payload["header"]["model"]["model"] != CHALLENGER_MODEL:
        problems.append(f"{key}: row stamped with {payload['header']['model']['model']}")
missing = sorted(set(SMOKE) - {k for k, _ in smoke_rows})
if missing:
    problems.append(f"no row for {missing}")
if problems:
    print("\n".join(problems))
    raise SystemExit("CHECKPOINT 2 FAILED: the challenger is not compatible as configured. Record this as a compatibility failure; do not retry with different budgets in this notebook.")
SMOKE_OK = True
print("CHECKPOINT 2 passed: the challenger runs the unchanged D1 v3 investigator")

## 6. Challenger: full 20-case run

Resumable exactly like the baseline within this runtime; the export in section 8 (or a `download_zip` call) brings the rows home.

In [ ]:
if not SMOKE_OK:
    raise SystemExit("pass CHECKPOINT 2 first")
result = run("run", "--model", CHALLENGER_MODEL, "--arm", ARM, "--repeat", str(REPEAT), "--seed", str(SEED), check=False)
print("run exit code", result.returncode)
run("summarise", "--model", CHALLENGER_MODEL, "--arm", ARM, "--repeat", str(REPEAT))
summary_ch = json.loads((DEV_DIR / f"SUMMARY_{ARM}_{la.model_slug(CHALLENGER_MODEL)}_rep{REPEAT}.json").read_text())
print(f"rows written {summary_ch['rows_written']}/{EXPECTED_CASES}; completed strict {summary_ch['completed_strict']}; excluded {summary_ch.get('rows_excluded')}")
save_results(CHALLENGER_MODEL, RESULTS_CHALLENGER)
if summary_ch["rows_written"] != EXPECTED_CASES or summary_ch.get("rows_excluded"):
    raise SystemExit("the challenger run is not complete; re-run this cell after a disconnect (finished rows are skipped)")
CHALLENGER_OK = True
print("challenger complete")

## 7. Comparison

`scripts/local_compare.py` pairs the two row sets by case, refuses if anything but the model differs (manifest, prompt version, investigator hashes, client configuration, repeat, seed), and writes `COMPARE_*.md/json`. The orderings it applies are stated in its header; it computes no significance test.

In [ ]:
if not (BASELINE_OK and CHALLENGER_OK):
    raise SystemExit("both runs must be complete and checkpointed")
subprocess.run([sys.executable, "scripts/local_compare.py", "--arm", ARM, "--models", BASELINE_MODEL, CHALLENGER_MODEL, "--repeat", str(REPEAT)], check=True)
stem = f"COMPARE_{ARM}_{la.model_slug(BASELINE_MODEL)}_vs_{la.model_slug(CHALLENGER_MODEL)}_rep{REPEAT}"
from IPython.display import Markdown, display
display(Markdown((DEV_DIR / f"{stem}.md").read_text()))

In [ ]:
# Both summaries side by side, for the numbers the comparison table abbreviates.
for model in (BASELINE_MODEL, CHALLENGER_MODEL):
    text = (DEV_DIR / f"SUMMARY_{ARM}_{la.model_slug(model)}_rep{REPEAT}.md").read_text()
    print(text[: text.index("## Per case")] if "## Per case" in text else text)

## 8. Export

Two downloads. **Results**: rows, freezes, summaries, the comparison and a session record; upload this zip in section 1 next time and every validated row is skipped. **Telemetry cache** (optional, ~1.6 GB): `data/external` with a checksum file; upload it next time and section 2 verifies it and skips the fetch. Raw telemetry is not in the results zip; rows carry event ids and short rendered lines only.

In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_dir = SUMMARIES / f"{MANIFEST_HASH[:12]}_{stamp}"
export_dir.mkdir(parents=True, exist_ok=True)
for p in DEV_DIR.iterdir():
    if p.is_file() and p.suffix in (".json", ".md") and p.name.startswith(("MANIFEST", "ENVIRONMENT", "SUMMARY", "COMPARE", "D1_")):
        shutil.copy2(p, export_dir / p.name)
machine = {"gpus": gpus, "ram_available_bytes": mem_available_bytes(), "ollama_version": json.load(urllib.request.urlopen(OLLAMA_URL + "/api/version"))["version"],
           "head": head, "manifest_hash": MANIFEST_HASH, "baseline": BASELINE_MODEL, "challenger": CHALLENGER_MODEL, "exported_at": stamp}
(export_dir / "SESSION.json").write_text(json.dumps(machine, indent=1))

os.chdir("/content")
download_zip(f"ath_results_{MANIFEST_HASH[:12]}_{stamp}", Path("agentic-threat-hunter/reports/local/dev"), Path("results"))
os.chdir(REPO)

In [ ]:
# Optional: the telemetry cache, so the next session skips the fetch. Structure: telemetry_cache/CHECKSUMS.json
# + telemetry_cache/data_external/{flaws_cloud,dedale}. Section 2 verifies sizes and hashes before reuse.
if EXPORT_TELEMETRY_CACHE:
    cache_root = Path("/content/telemetry_cache")
    if cache_root.exists():
        shutil.rmtree(cache_root)
    (cache_root / "data_external").mkdir(parents=True)
    for sub in CACHED_SUBDIRS:
        subprocess.run(["rsync", "-a", str(EXTERNAL / sub) + "/", str(cache_root / "data_external" / sub) + "/"], check=True)
    files_ = checksum_tree(cache_root / "data_external")
    (cache_root / "CHECKSUMS.json").write_text(json.dumps({
        "source": "data/external as fetched by scripts/fetch_external.py flaws_cloud and scripts/dedale_fetch_hours.py --days 2",
        "subdirs": CACHED_SUBDIRS, "files": files_}, indent=1))
    os.chdir("/content")
    download_zip(f"telemetry_cache_{stamp}", cache_root)
    os.chdir(REPO)
else:
    print("EXPORT_TELEMETRY_CACHE is False; nothing exported")